In [66]:
%load_ext autoreload
%autoreload 2
    
import re
import os
import sys
import random
import math
import matplotlib.pyplot as plt

from collections import Counter, defaultdict
from tqdm import tqdm_notebook
from glob import glob

sys.path.append("../src")

from word_order.process_treebank import create_word_order_df, read_df


from multiblimp.languages import get_ud_langs

from word_order.create_pairs import create_pairs
from word_order.prediction_target import *
from word_order.process_treebank import load_treebank
from word_order.decision_tree import fit_dt
from word_order.entropy import order_entropy
from word_order.viz_tree import tree2html
from word_order.utils import MAX_TREEBANK_LEN

import numpy as np

import pandas as pd


random.seed(42)
resource_dir = "/media/jaap/81b6ce8a-28e5-4eda-9c68-b13e0637cc4f/WORD_ORDER/resources"

target = obl_target

deprel_dir = "_".join(target.child_deprels)
dt_df_dir = f"/media/jaap/81b6ce8a-28e5-4eda-9c68-b13e0637cc4f/WORD_ORDER/dt_df/{deprel_dir}"
pair_dir = f"/media/jaap/81b6ce8a-28e5-4eda-9c68-b13e0637cc4f/WORD_ORDER/pairs/{deprel_dir}"
word_order_dir = f"/media/jaap/81b6ce8a-28e5-4eda-9c68-b13e0637cc4f/WORD_ORDER/treebank_features/{deprel_dir}"

langs = [path.split('/')[-1].split('.')[0] for path in glob(word_order_dir+"/*.csv")]

predictor_var = "deprel_order"

# langs = ["Dutch"]
lang2data = {}


for lang in tqdm_notebook(sorted(langs), total=len(langs)):
    # lang = " ".join(fn.split("_")[:-2]) or fn
    html_file = f"word_order/decision_trees/html/{deprel_dir}/{lang}.html"

    if os.path.exists(os.path.join(pair_dir, f"{lang}.csv")):
        print('skipping', lang)
        continue
    # if lang in lang2data:
    #     continue
    
    print(lang)
    raw_df = read_df(lang, word_order_dir=word_order_dir)
    
    full_df = raw_df[raw_df[predictor_var].notnull()]

    if len(full_df) == 0:
        continue

    # subset core_arg df
    # if 'nsubj_sibling-deprel_aux' in full_df.columns:
    #     full_df = full_df[~full_df['nsubj_sibling-deprel_aux']]
    # if 'nsubj_sibling-deprel_cop' in full_df.columns:
    #     full_df = full_df[~full_df['nsubj_sibling-deprel_cop']]
    # if 'head_Tense' in full_df.columns:
    #     full_df = full_df[~full_df.head_Tense.isna()]

    omit_feats = {col for col in full_df.columns if ('form' in col) or ('lemma' in col)}
    omit_feats.add('core_args')

    min_impurity_decrease = get_impurity(len(full_df))
    
    model, dt_df, predictor_df = fit_dt(
        full_df, 
        target,
        verbose=1, 
        min_impurity_decrease=min_impurity_decrease,
        min_samples_leaf=10,
        save_to=os.path.join(dt_df_dir, lang),
        omit_feats=omit_feats,
    )

    if model is None:
        print("skipping", lang)
        continue

    balance_features = ["sen_len", "head_form"]
    balance_features += [f"{deprel}_form" for deprel in target.child_deprels]
    
    treebank = load_treebank(lang, resource_dir, max_treebank_len=MAX_TREEBANK_LEN)
    swap_df = create_pairs(
        dt_df,
        treebank, 
        target,
        balance_features=balance_features,
        save_to=os.path.join(pair_dir, f"{lang}.csv"),
    )

    tree2html(
        model, 
        dt_df, 
        full_df, 
        predictor_var,
        target,
        html_file, 
        max_rows=15,
        meta={"Language": lang},
        only_show_real_orders=True,
        correlate_features=True,
    )

    lang2data[lang] = model, dt_df

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


/tmp/ipykernel_360929/919227429.py:53: TqdmDeprecationWarning:

This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`



  0%|          | 0/171 [00:00<?, ?it/s]

Abaza
ERROR! Session/line number was not unique in database. History logging moved to new session 405
Train acc 0.8809523809523809
Test acc  1.0
Abkhaz


/home/jaap/gpu/lib/python3.13/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning:

Found unknown categories in columns [0, 6, 33, 45, 52, 57, 86] during transform. These unknown categories will be encoded as all zeros



...

/home/jaap/gpu/lib/python3.13/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning:

Found unknown categories in columns [22, 28, 49] during transform. These unknown categories will be encoded as all zeros

<string>:10: FutureWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



Train acc 0.9355932203389831
Test acc  0.8484848484848485
Yupik
Train acc 0.8448275862068966
Test acc  0.7142857142857143


/home/jaap/gpu/lib/python3.13/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning:

Found unknown categories in columns [37, 45, 48, 51, 71, 76, 88, 97, 112] during transform. These unknown categories will be encoded as all zeros

/home/jaap/gpu/lib/python3.13/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning:

Found unknown categories in columns [25, 32, 33] during transform. These unknown categories will be encoded as all zeros



Zaar
Train acc 0.8402777777777778
Test acc  0.6875


/home/jaap/gpu/lib/python3.13/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning:

Found unknown categories in columns [37] during transform. These unknown categories will be encoded as all zeros



In [63]:
%load_ext autoreload
%autoreload 2
    
import os
import ast
from tqdm import tqdm_notebook
from glob import glob

sys.path.append("../src")

from multiblimp.languages import get_ud_langs

from word_order.create_pairs import create_pairs
from word_order.prediction_target import *
from word_order.process_treebank import load_treebank


import pandas as pd


random.seed(42)

target = amod_target

langs = [path.split('/')[-1].split('.')[0] for path in glob(word_order_dir+"/*.csv")]
# langs = ["Dutch"]


for lang in tqdm_notebook(sorted(langs), total=len(langs)):
    dt_df = pd.read_csv(
        os.path.join(dt_df_dir, f"{lang}.csv"),
        converters={
            "sen": ast.literal_eval, 
            "swap_order_candidates": ast.literal_eval
        },
    )
    
    balance_features = ["sen_len", "head_form"]
    balance_features += [f"{deprel}_form" for deprel in target.child_deprels]
    
    treebank = load_treebank(lang, resource_dir, max_treebank_len=MAX_TREEBANK_LEN)
    swap_df = create_pairs(
        dt_df,
        treebank, 
        target,
        balance_features=balance_features,
        save_to=os.path.join(pair_dir, f"{lang}.csv"),
    )

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


/tmp/ipykernel_360929/918998162.py:29: TqdmDeprecationWarning:

This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`



  0%|          | 0/173 [00:00<?, ?it/s]

1000


KeyError: 'amod_form'

In [ ]:
from word_order.huggingface import upload


pair_dir = f"/media/jaap/81b6ce8a-28e5-4eda-9c68-b13e0637cc4f/WORD_ORDER/pairs"
upload(pair_dir, "jumelet/multiblimp-word-order")

Loading pairs from /media/jaap/81b6ce8a-28e5-4eda-9c68-b13e0637cc4f/WORD_ORDER/pairs ...


Uploading subsets:   0%|                                | 0/119 [00:00<?, ?it/s]

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

                                        : 100%|##########| 23.6kB / 23.6kB            

...

README.md: 0.00B [00:00, ?B/s]

Uploading subsets:  97%|█████████████████████▎| 115/119 [08:52<00:19,  4.95s/it]

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

                                        : 100%|##########| 23.0kB / 23.0kB            

In [69]:
import lm_eval

lm_eval.__version__

'0.4.11'

In [71]:
from datasets import load_dataset

ds = load_dataset("jumelet/multiblimp-word-order", "Dutch")

Dutch/advmod-00000-of-00001.parquet:   0%|          | 0.00/145k [00:00<?, ?B/s]

Dutch/amod-00000-of-00001.parquet:   0%|          | 0.00/253k [00:00<?, ?B/s]

Dutch/nsubj_obj-00000-of-00001.parquet:   0%|          | 0.00/227k [00:00<?, ?B/s]

Dutch/obl-00000-of-00001.parquet:   0%|          | 0.00/89.3k [00:00<?, ?B/s]

Generating advmod split:   0%|          | 0/667 [00:00<?, ? examples/s]

Generating amod split:   0%|          | 0/999 [00:00<?, ? examples/s]

Generating nsubj_obj split:   0%|          | 0/3870 [00:00<?, ? examples/s]

Generating obl split:   0%|          | 0/488 [00:00<?, ? examples/s]

In [72]:
ds

DatasetDict({
    advmod: Dataset({
        features: ['leaf_rule', 'sen_str', 'original_order', 'swap_order', 'swapped_sen_str'],
        num_rows: 667
    })
    amod: Dataset({
        features: ['leaf_rule', 'sen_str', 'original_order', 'swap_order', 'swapped_sen_str'],
        num_rows: 999
    })
    nsubj_obj: Dataset({
        features: ['leaf_rule', 'sen_str', 'original_order', 'swap_order', 'swapped_sen_str'],
        num_rows: 3870
    })
    obl: Dataset({
        features: ['leaf_rule', 'sen_str', 'original_order', 'swap_order', 'swapped_sen_str'],
        num_rows: 488
    })
})

In [64]:
Counter(swap_df[swap_df.keep].amod_form).most_common(10)

[('Zorggerichte', 1),
 ('avondlijke', 1),
 ('West-Duitse', 1),
 ('Oosteuropese', 1),
 ('onbekend', 1),
 ('ouderwetse', 1),
 ('Zuidafrikaanse', 1),
 ('Bloemendaalse', 1),
 ('fiscaal', 1),
 ('aanstaande', 1)]

In [30]:
row = swap_df.iloc[0]

print(row["Na_sen_str"])
print(row["aN_sen_str"])

Donderdag won Lodewijk Meeter met Siete Meeter op de plaats tweede;
Donderdag won Lodewijk Meeter met Siete Meeter op de tweede plaats;


In [17]:
row = swap_df.iloc[0]

print(row["Nn_sen_str"])
print(row["nN_sen_str"])

Hij stuurde zijn minister van de Vrede naar een bespreking met de Amerikaanse president George W. Bush om bezwaren tegen die oorlog over te brengen, zoals hij dit ook al gedaan had bij de Eerste Golfoorlog in 1991.
Hij stuurde zijn minister van de Vrede naar een bespreking met de Amerikaanse president George W. Bush om tegen die oorlog bezwaren over te brengen, zoals hij dit ook al gedaan had bij de Eerste Golfoorlog in 1991.


In [27]:
from word_order.viz_tree import get_sample_ids

prep = model.named_steps["preprocessor"]
clf = model.named_steps["clf"]

predictor_value = 'Nn'

sample_ids = get_sample_ids(prep, clf, dt_df, predictor_var, max_rows=15)

# Pick the predictor_value where you saw the bug
# Check node 2
node_2_ids = sample_ids[predictor_value][2]  # adjust predictor_value accordingly

X_check = prep.transform(dt_df.loc[node_2_ids])
paths = clf.decision_path(X_check)
visits_node_2 = paths[:, 2].toarray().flatten().astype(bool)

print("All visit node 2?", visits_node_2.all())
print("Count that don't:", (~visits_node_2).sum())


All visit node 2? True
Count that don't: 0


In [36]:
for idx, ids in sample_ids['nN'].items():
    for jdx in ids:
        print(idx, dt_df.loc[jdx].leaf_id)

0 3
0 3
0 3
0 3
0 2
0 2
0 4
0 4
0 2
0 2
0 2
0 2
0 2
0 3
0 3
1 3
1 3
1 3
1 3
1 3
1 3
1 2
1 3
1 3
1 2
1 3
1 3
1 2
1 2
1 2
2 2
2 2
2 2
2 2
2 2
2 2
2 2
2 2
2 2
2 2
2 2
2 2
2 2
2 2
2 2
3 3
3 3
3 3
3 3
3 3
3 3
3 3
3 3
3 3
3 3
3 3
3 3
3 3
3 3
3 3
4 4
4 4
4 4
4 4
4 4
4 4
4 4
4 4
4 4
4 4
4 4
4 4
4 4
4 4
4 4


In [44]:
for idx, row in dt_df.iterrows():
    # if row.sen == ['Dit', 'soort', 'verhalen', 'berusten', 'echter', 'zelden', 'op', 'bewijzen', '.']:
    if "duikbootmotor" in row.sen:
        print(row.sen, row.leaf_id, idx, row['nmod_child-feat_det_Definite'], full_df.loc[idx].nmod_form, full_df.loc[idx].head_form)
        # break


['Het', 'geval', 'van', 'dertig', 'meter', 'lengte', 'en', '300', 'ton', 'gewicht', '(', 'door', 'een', 'pantsering', 'van', '76', 'mm', 'rondom', ')', 'moest', 'voorzien', 'worden', 'van', 'drie', 'gevechtstorens', ',', 'ieder', 'bewapend', 'met', 'twee', '102', 'mm', 'kanonnen', 'en', 'voortgestuwd', 'worden', 'door', 'een', '800', 'pk', 'duikbootmotor', '.'] 4 5133 Ind pantsering ton
['Het', 'geval', 'van', 'dertig', 'meter', 'lengte', 'en', '300', 'ton', 'gewicht', '(', 'door', 'een', 'pantsering', 'van', '76', 'mm', 'rondom', ')', 'moest', 'voorzien', 'worden', 'van', 'drie', 'gevechtstorens', ',', 'ieder', 'bewapend', 'met', 'twee', '102', 'mm', 'kanonnen', 'en', 'voortgestuwd', 'worden', 'door', 'een', '800', 'pk', 'duikbootmotor', '.'] 4 5131 _missing meter geval
['Het', 'geval', 'van', 'dertig', 'meter', 'lengte', 'en', '300', 'ton', 'gewicht', '(', 'door', 'een', 'pantsering', 'van', '76', 'mm', 'rondom', ')', 'moest', 'voorzien', 'worden', 'van', 'drie', 'gevechtstorens', ',

In [34]:
dt_df.loc[idx].leaf_id

4

In [45]:
row['nmod_child-feat_det_Definite']

'Def'

In [6]:
from word_order.viz_deprel import generate_html_deprel_index


deprel = "_".join(target.child_deprels)
# deprel = "amod"

generate_html_deprel_index(
    f"/media/jaap/81b6ce8a-28e5-4eda-9c68-b13e0637cc4f/WORD_ORDER/dt_df/{deprel}", 
    f"word_order/decision_trees/html/{deprel}/",
    language_data=lang2data,
)

100%|█████████████████████████████████████████| 157/157 [01:15<00:00,  2.09it/s]


In [7]:
from word_order.viz_overview import generate_html_overview_index


generate_html_overview_index('word_order/decision_trees/html/')

In [17]:
from tqdm import tqdm


rows = []

for _, row in tqdm(full_df.iterrows()):
    for order in row.swap_order_candidates:
        rows.append((row.language, row.sen_str, row[f"{order}_sen_str"], row.core_args, order))

swap_df = pd.DataFrame(rows, columns=["language", "sen", "swap_sen", "sen_order", "swap_order"])
swap_df.to_csv("word_order/pairs/all_pairs.csv", index=False)

In [20]:
import pickle

with open("lang2data.pickle", "wb") as f:
    pickle.dump(lang2data, f)

full_df.to_csv("full_df.csv", index=False)

In [63]:
threshold = 0.1
swap_so = str.maketrans({"s": "o", "o": "s"})

correct_num_swaps = []
all_swap_order_candidates = []

all_orders = {"svo", "ovs", "osv", "sov", "vos", "vso"}

for _, row in dt_df.iterrows():
    core_arg = row.core_args
    swap_orders = all_orders - {core_arg, core_arg.translate(swap_so)}

    swap_order_candidates = [
        arg_order
        for arg_order in swap_orders
        if row[f"{arg_order}_entropy"] < threshold
    ]
    
    num_swaps = len(swap_order_candidates)
    
    correct_num_swaps.append(num_swaps)
    all_swap_order_candidates.append(swap_order_candidates)

dt_df["num_swaps"] = correct_num_swaps
dt_df["swap_order_candidates"] = all_swap_order_candidates

In [75]:
(swap_df["num_swaps"] > 0).mean()

0.0

In [49]:
row[[
    f"{arg_order}_entropy"
    for arg_order in swap_orders
]] < 0.04

vso_entropy    False
osv_entropy     True
sov_entropy     True
vos_entropy     True
Name: 996, dtype: bool

In [22]:
generate_html_index('word_order/decision_trees/html/')

In [24]:
"core_args" in raw_df.columns

False

In [37]:
"this Is a test".capitalize()

'This is a test'